# 🎬 Video Object Replacement Pipeline

## Архитектура

```
Локально (твоя GPU):
  → GroundingDINO:  детекция объекта на первом кадре
  → SAM2:           сегментация + трекинг масок
  → Shot detector:  обнаружение смены камеры
  → Re-ID (CLIP):   переопознавание объекта после исчезновения
  → Optical flow:   RAFT для варпинга
  → Compositing:    склейка результата обратно в кадр

API (Replicate):
  → Wan2.1-Fun-Inpaint или SDXL Inpainting
  → вызывается только при: старте, смене камеры, re-появлении объекта
  → вход: bbox-crop кадра + маска + промпт
  → выход: замена объекта с сохранением фона
```

## Edge cases которые обрабатываются
- ✅ Смена камеры (shot cut) → re-detect + новый inpainting
- ✅ Объект пропал из кадра → пауза трекинга
- ✅ Объект вернулся → re-ID через CLIP + новый inpainting
- ✅ Плавное движение → warping без API вызова
- ✅ Сильное движение → conditional regeneration

## 1. Установка зависимостей

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu118')
pip_install('opencv-python', 'numpy', 'matplotlib', 'tqdm', 'Pillow')
pip_install('transformers', 'accelerate')
pip_install('replicate')  # API клиент

# GroundingDINO
subprocess.run(['git', 'clone', 'https://github.com/IDEA-Research/GroundingDINO.git'], check=False)
pip_install('-e', 'GroundingDINO')

# SAM2
subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam2.git'], check=False)
pip_install('-e', 'sam2')

# RAFT optical flow
subprocess.run(['git', 'clone', 'https://github.com/princeton-vl/RAFT.git'], check=False)

print('✅ Зависимости установлены')

## 2. Конфигурация

Все настраиваемые параметры в одном месте.

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Config:
    # --- Пути ---
    input_video_path: str = 'input.mp4'
    output_video_path: str = 'output.mp4'
    frames_dir: str = 'frames/'
    output_frames_dir: str = 'output_frames/'
    weights_dir: str = 'models/'

    # --- Промпт ---
    detection_prompt: str = 'coffee cup'          # что детектировать
    replacement_prompt: str = 'white ceramic mig with red text "ALFA"'
    negative_prompt: str = 'blurry, deformed, ugly'

    # --- API ---
    replicate_api_token: str = 'YOUR_REPLICATE_TOKEN'
    # Модель на Replicate: wan2.1 inpainting или sdxl inpainting
    replicate_model: str = 'wan-video/wan2.1-fun-inpaint'
    use_video_model: bool = True   # True = видео модель, False = картиночная
    video_chunk_size: int = 16     # кадров в одном вызове видео модели

    # --- Детекция и трекинг ---
    grounding_dino_config: str = 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
    grounding_dino_weights: str = 'weights/groundingdino_swint_ogc.pth'
    sam2_config: str = 'sam2/configs/sam2.1/sam2.1_hiera_large.yaml'
    sam2_weights: str = 'weights/sam2.1_hiera_large.pt'
    raft_weights: str = 'weights/raft-things.pth'

    # --- Пороги ---
    # Shot cut detection
    HIST_THRESHOLD: float = 0.4       # порог разницы гистограмм для shot cut
    CUT_IOU_THRESHOLD: float = 0.2    # IoU ниже этого = shot cut

    # Regeneration
    FLOW_THRESHOLD: float = 15.0      # пиксели/кадр, выше = regenerate
    IOU_THRESHOLD: float = 0.5        # IoU ниже = regenerate

    # Re-identification
    REID_CLIP_THRESHOLD: float = 0.75 # CLIP similarity для подтверждения re-ID
    MAX_FRAMES_MISSING: int = 60      # кадров без объекта → сброс трекинга

    # Bbox padding
    BBOX_PADDING: float = 0.15        # 15% padding вокруг bbox

    # --- Видео ---
    target_fps: Optional[int] = None  # None = сохранить оригинальный fps
    max_frames: Optional[int] = None  # None = весь видос

    # --- Воспроизводимость ---
    seed: int = 42

    # --- Устройство ---
    device: str = 'cuda'

cfg = Config()

os.makedirs(cfg.frames_dir, exist_ok=True)
os.makedirs(cfg.output_frames_dir, exist_ok=True)
os.makedirs(cfg.weights_dir, exist_ok=True)

print('✅ Конфигурация загружена')
print(f'   Детектируем: "{cfg.detection_prompt}"')
print(f'   Заменяем на: "{cfg.replacement_prompt}"')

## 3. Импорты и VideoState

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import replicate
import base64
import io
import sys
import os
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, List, Tuple
from PIL import Image
from tqdm import tqdm
from copy import deepcopy

os.environ['REPLICATE_API_TOKEN'] = cfg.replicate_api_token

# ─── Главный класс состояния пайплайна ───────────────────────────────────────

@dataclass
class VideoState:
    """Хранит всё промежуточное состояние пайплайна."""

    # Основные данные
    frames: List[np.ndarray] = field(default_factory=list)         # BGR кадры
    masks: List[Optional[np.ndarray]] = field(default_factory=list) # бинарные маски
    flows: List[Optional[np.ndarray]] = field(default_factory=list) # optical flow
    bboxes: List[Optional[Tuple]] = field(default_factory=list)     # (x1,y1,x2,y2)
    output_frames: List[np.ndarray] = field(default_factory=list)

    # Inpainting результаты: индекс кадра → crop с заменой
    generated_objects: dict = field(default_factory=dict)

    # Shot cut / исчезновение объекта
    shot_boundaries: List[int] = field(default_factory=list)   # индексы cut кадров
    object_visible: List[bool] = field(default_factory=list)   # виден ли объект
    frames_missing: int = 0                                    # счётчик пропущенных кадров

    # Re-identification
    last_known_appearance: Optional[torch.Tensor] = None  # CLIP эмбеддинг объекта
    last_known_bbox: Optional[Tuple] = None
    last_known_mask: Optional[np.ndarray] = None

    # Мета
    fps: float = 25.0
    frame_size: Tuple[int, int] = (1280, 720)  # (w, h)

    def add_frame(self, frame, mask=None, bbox=None, visible=True):
        self.frames.append(frame)
        self.masks.append(mask)
        self.bboxes.append(bbox)
        self.object_visible.append(visible)

print('✅ Импорты выполнены')

## 4. Загрузка весов моделей

Скачиваем веса GroundingDINO, SAM2 и RAFT если их нет.

In [ ]:
import urllib.request

WEIGHTS = {
    cfg.grounding_dino_weights: 'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth',
    cfg.sam2_weights: 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt',
    cfg.raft_weights: 'https://dl.dropboxusercontent.com/s/4j4z58wuv8o0mfz/models.zip',  # RAFT weights zip
}

def download_weights(weights_dict):
    for path, url in weights_dict.items():
        if not os.path.exists(path):
            print(f'Загружаем {path}...')
            urllib.request.urlretrieve(url, path)
            print(f'  ✅ {path}')
        else:
            print(f'  ✓ {path} уже есть')

download_weights({k: v for k, v in WEIGHTS.items() if 'raft' not in k})
# RAFT качаем отдельно (zip)
if not os.path.exists(cfg.raft_weights):
    import zipfile
    urllib.request.urlretrieve(WEIGHTS[cfg.raft_weights], 'weights/raft_models.zip')
    with zipfile.ZipFile('weights/raft_models.zip', 'r') as z:
        z.extractall('weights/')
    print('✅ RAFT веса распакованы')

## 5. Загрузка видео и извлечение кадров

In [ ]:
def load_video(video_path: str, max_frames: Optional[int] = None) -> Tuple[List[np.ndarray], float, Tuple]:
    """Загружает видео и возвращает (frames, fps, (width, height))."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Не могу открыть видео: {video_path}')

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'📹 Видео: {total} кадров, {fps:.1f} fps, {w}x{h}')

    frames = []
    limit = max_frames or total

    for _ in tqdm(range(limit), desc='Извлекаем кадры'):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)

    cap.release()
    print(f'✅ Загружено {len(frames)} кадров')
    return frames, fps, (w, h)


def save_frames(frames: List[np.ndarray], directory: str):
    """Сохраняет кадры в папку для отладки."""
    os.makedirs(directory, exist_ok=True)
    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(directory, f'frame_{i:05d}.jpg'), frame)


# Загружаем видео в state
state = VideoState()
raw_frames, fps, frame_size = load_video(cfg.input_video_path, cfg.max_frames)
state.fps = fps
state.frame_size = frame_size
state.frames = raw_frames

## 6. Детектор смены камеры (Shot Cut Detection)

Используем два признака:
1. **Разница гистограмм** — резкая смена цветового распределения
2. **IoU масок** — объект резко сместился

При обнаружении cut — помечаем кадр как границу сцены.

In [ ]:
def compute_histogram_diff(frame1: np.ndarray, frame2: np.ndarray) -> float:
    """Разница гистограмм HSV между кадрами. Больше = сильнее изменение."""
    hsv1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2HSV)
    hsv2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2HSV)

    scores = []
    for channel in range(3):
        h1 = cv2.calcHist([hsv1], [channel], None, [64], [0, 256])
        h2 = cv2.calcHist([hsv2], [channel], None, [64], [0, 256])
        cv2.normalize(h1, h1)
        cv2.normalize(h2, h2)
        # Correlation: 1.0 = идентично, -1.0 = противоположно
        score = cv2.compareHist(h1, h2, cv2.HISTCMP_CORREL)
        scores.append(score)

    # Возвращаем «несхожесть»: 0 = одинаково, 1 = полностью разные
    return 1.0 - float(np.mean(scores))


def compute_mask_iou(mask1: Optional[np.ndarray], mask2: Optional[np.ndarray]) -> float:
    """IoU двух бинарных масок. None маска → IoU = 0."""
    if mask1 is None or mask2 is None:
        return 0.0
    m1 = mask1.astype(bool)
    m2 = mask2.astype(bool)
    intersection = (m1 & m2).sum()
    union = (m1 | m2).sum()
    return float(intersection / union) if union > 0 else 0.0


def detect_shot_cut(
    prev_frame: np.ndarray,
    curr_frame: np.ndarray,
    prev_mask: Optional[np.ndarray],
    curr_mask: Optional[np.ndarray],
    hist_threshold: float,
    iou_threshold: float
) -> bool:
    """
    Возвращает True если между кадрами произошла смена камеры.
    Используем два независимых признака — любой из них может сигнализировать cut.
    """
    hist_diff = compute_histogram_diff(prev_frame, curr_frame)
    if hist_diff > hist_threshold:
        return True

    # Проверяем IoU только если обе маски есть
    if prev_mask is not None and curr_mask is not None:
        iou = compute_mask_iou(prev_mask, curr_mask)
        if iou < iou_threshold:
            return True

    return False


def precompute_shot_boundaries(
    frames: List[np.ndarray],
    hist_threshold: float
) -> List[int]:
    """
    Предварительный проход по видео для нахождения всех shot cuts.
    Работает только по гистограммам (маски ещё не известны).
    """
    boundaries = [0]  # первый кадр всегда граница
    for i in tqdm(range(1, len(frames)), desc='Ищем смены камеры'):
        diff = compute_histogram_diff(frames[i-1], frames[i])
        if diff > hist_threshold:
            boundaries.append(i)

    print(f'🎬 Найдено {len(boundaries)} сцен(ы): кадры {boundaries}')
    return boundaries


# Предварительно находим все смены камеры
state.shot_boundaries = precompute_shot_boundaries(state.frames, cfg.HIST_THRESHOLD)

## 7. GroundingDINO — детекция объекта по тексту

Детектируем объект на первом кадре каждой сцены.

In [ ]:
sys.path.insert(0, 'GroundingDINO')
from groundingdino.util.inference import load_model as load_gdino, load_image, predict as gdino_predict
from groundingdino.util import box_ops

def load_grounding_dino(config_path: str, weights_path: str, device: str):
    """Загружает модель GroundingDINO."""
    model = load_gdino(config_path, weights_path)
    model = model.to(device).eval()
    print('✅ GroundingDINO загружен')
    return model


def detect_object(
    model,
    frame: np.ndarray,
    prompt: str,
    device: str,
    box_threshold: float = 0.35,
    text_threshold: float = 0.25
) -> Optional[Tuple[int, int, int, int]]:
    """
    Детектирует объект по текстовому промпту.
    Возвращает bbox (x1, y1, x2, y2) в пикселях или None если не найден.
    """
    h, w = frame.shape[:2]

    # GroundingDINO ожидает RGB PIL
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # Трансформируем как ожидает модель
    import torchvision.transforms as T
    transform = T.Compose([
        T.Resize([800], max_size=1333),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image_tensor = transform(pil_image).to(device)

    boxes, logits, phrases = gdino_predict(
        model=model,
        image=image_tensor,
        caption=prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
    )

    if len(boxes) == 0:
        print(f'  ⚠️ Объект "{prompt}" не найден в кадре')
        return None

    # Берём bbox с наибольшей уверенностью
    best_idx = logits.argmax().item()
    cx, cy, bw, bh = boxes[best_idx].tolist()

    # Конвертируем из нормализованных cxcywh → пиксели xyxy
    x1 = int((cx - bw / 2) * w)
    y1 = int((cy - bh / 2) * h)
    x2 = int((cx + bw / 2) * w)
    y2 = int((cy + bh / 2) * h)

    # Клипуем к границам кадра
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    print(f'  ✅ Найден "{phrases[best_idx]}" bbox=({x1},{y1},{x2},{y2}) conf={logits[best_idx]:.2f}')
    return (x1, y1, x2, y2)


def add_bbox_padding(bbox: Tuple, frame_shape: Tuple, padding: float) -> Tuple:
    """
    Добавляет padding вокруг bbox чтобы модель видела контекст.
    padding=0.15 означает расширение на 15% от каждой стороны.
    """
    x1, y1, x2, y2 = bbox
    h, w = frame_shape[:2]
    bw, bh = x2 - x1, y2 - y1
    pad_x = int(bw * padding)
    pad_y = int(bh * padding)

    x1p = max(0, x1 - pad_x)
    y1p = max(0, y1 - pad_y)
    x2p = min(w, x2 + pad_x)
    y2p = min(h, y2 + pad_y)

    return (x1p, y1p, x2p, y2p)


gdino_model = load_grounding_dino(cfg.grounding_dino_config, cfg.grounding_dino_weights, cfg.device)

## 8. SAM2 — сегментация и трекинг масок

SAM2 умеет трекать объект по видео. Мы:
1. Инициализируем трекер bbox с первого кадра сцены
2. Прогоняем SAM2 по кадрам сцены до следующего shot cut
3. При cut — реинициализируем трекер

In [ ]:
sys.path.insert(0, 'sam2')
from sam2.build_sam import build_sam2_video_predictor

def load_sam2(config_path: str, weights_path: str, device: str):
    """Загружает SAM2 video predictor."""
    predictor = build_sam2_video_predictor(config_path, weights_path, device=device)
    print('✅ SAM2 загружен')
    return predictor


def track_masks_in_scene(
    predictor,
    frames: List[np.ndarray],
    start_bbox: Tuple[int, int, int, int],
    start_frame_idx: int,
    end_frame_idx: int
) -> Tuple[List[Optional[np.ndarray]], List[Optional[Tuple]]]:
    """
    Трекает объект через SAM2 в диапазоне кадров [start_frame_idx, end_frame_idx).
    Возвращает (masks, bboxes) для каждого кадра в диапазоне.
    """
    scene_frames = frames[start_frame_idx:end_frame_idx]
    n = len(scene_frames)
    masks = [None] * n
    bboxes = [None] * n

    if n == 0:
        return masks, bboxes

    # SAM2 инициализация на первом кадре сцены
    with torch.inference_mode(), torch.autocast(cfg.device, dtype=torch.bfloat16):
        inference_state = predictor.init_state(video_path=None, offload_video_to_cpu=True)

        # Загружаем кадры напрямую как numpy
        inference_state['frames'] = [
            torch.from_numpy(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0
            for f in scene_frames
        ]

        x1, y1, x2, y2 = start_bbox
        _, obj_ids, mask_logits = predictor.add_new_points_or_box(
            inference_state,
            frame_idx=0,
            obj_id=1,
            box=np.array([x1, y1, x2, y2], dtype=np.float32)
        )

        # Прогоняем трекинг по всем кадрам сцены
        for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
            if len(mask_logits) == 0:
                continue

            mask = (mask_logits[0] > 0.0).squeeze().cpu().numpy().astype(np.uint8) * 255
            masks[frame_idx] = mask

            # Вычисляем bbox из маски
            ys, xs = np.where(mask > 0)
            if len(xs) > 0:
                bboxes[frame_idx] = (xs.min(), ys.min(), xs.max(), ys.max())

    return masks, bboxes


sam2_predictor = load_sam2(cfg.sam2_config, cfg.sam2_weights, cfg.device)

## 9. Re-Identification через CLIP

Когда объект пропадает из кадра и появляется снова — нужно убедиться что это тот же объект.
Используем CLIP эмбеддинги crop'а объекта для сравнения.

In [ ]:
from transformers import CLIPProcessor, CLIPModel

def load_clip(device: str):
    """Загружает CLIP для re-identification."""
    model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
    processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    model.eval()
    print('✅ CLIP загружен')
    return model, processor


def get_object_embedding(
    clip_model,
    clip_processor,
    frame: np.ndarray,
    bbox: Tuple[int, int, int, int],
    device: str
) -> torch.Tensor:
    """Возвращает нормализованный CLIP эмбеддинг crop'а объекта."""
    x1, y1, x2, y2 = bbox
    crop = frame[y1:y2, x1:x2]
    pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

    inputs = clip_processor(images=pil_crop, return_tensors='pt').to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = F.normalize(emb, dim=-1)
    return emb


def compute_clip_similarity(
    emb1: torch.Tensor,
    emb2: torch.Tensor
) -> float:
    """Cosine similarity между двумя CLIP эмбеддингами."""
    return float((emb1 * emb2).sum())


def is_same_object(
    clip_model,
    clip_processor,
    frame: np.ndarray,
    new_bbox: Tuple,
    reference_embedding: torch.Tensor,
    threshold: float,
    device: str
) -> bool:
    """
    Проверяет является ли объект в new_bbox тем же объектом
    что был запомнен в reference_embedding.
    """
    new_emb = get_object_embedding(clip_model, clip_processor, frame, new_bbox, device)
    sim = compute_clip_similarity(reference_embedding, new_emb)
    print(f'  Re-ID CLIP similarity: {sim:.3f} (порог {threshold})')
    return sim >= threshold


clip_model, clip_processor = load_clip(cfg.device)

## 10. Optical Flow (RAFT)

Используем для двух целей:
1. **Варпинг** сгенерированного объекта между ключевыми кадрами
2. **Определение** нужна ли regeneration (по magnitude flow)

In [ ]:
sys.path.insert(0, 'RAFT/core')
from raft import RAFT
from utils.utils import InputPadder

def load_raft(weights_path: str, device: str):
    """Загружает RAFT optical flow модель."""
    import argparse
    args = argparse.Namespace(small=False, mixed_precision=False, alternate_corr=False)
    model = RAFT(args)
    state_dict = torch.load(weights_path, map_location=device)
    # Убираем префикс 'module.' если модель была обёрнута в DataParallel
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model = model.to(device).eval()
    print('✅ RAFT загружен')
    return model


def compute_optical_flow(
    raft_model,
    frame1: np.ndarray,
    frame2: np.ndarray,
    device: str
) -> np.ndarray:
    """
    Вычисляет optical flow из frame1 в frame2.
    Возвращает flow (H, W, 2) в пикселях.
    """
    def to_tensor(frame):
        return torch.from_numpy(frame).permute(2, 0, 1).float()[None].to(device)

    t1 = to_tensor(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
    t2 = to_tensor(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))

    padder = InputPadder(t1.shape)
    t1, t2 = padder.pad(t1, t2)

    with torch.no_grad():
        _, flow = raft_model(t1, t2, iters=20, test_mode=True)

    flow = padder.unpad(flow)
    return flow[0].permute(1, 2, 0).cpu().numpy()  # (H, W, 2)


def compute_flow_magnitude(flow: np.ndarray) -> float:
    """Средняя magnitude optical flow (пиксели/кадр)."""
    mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
    return float(mag.mean())


def warp_image_with_flow(
    image: np.ndarray,
    flow: np.ndarray
) -> np.ndarray:
    """
    Варпит image используя optical flow.
    flow имеет форму (H, W, 2) — смещения в пикселях.
    """
    h, w = flow.shape[:2]
    # Создаём сетку абсолютных координат куда должны попасть пиксели
    grid_y, grid_x = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x = (grid_x + flow[..., 0]).astype(np.float32)
    map_y = (grid_y + flow[..., 1]).astype(np.float32)
    warped = cv2.remap(image, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return warped


raft_model = load_raft(cfg.raft_weights, cfg.device)

## 11. Логика: regenerate vs warp

Каждый кадр мы решаем — вызывать API или сделать варпинг локально.

**Regenerate если:**
- Первый кадр сцены (всегда)
- Shot cut обнаружен
- Объект только что вернулся после отсутствия
- Flow magnitude > FLOW_THRESHOLD
- Mask IoU < IOU_THRESHOLD

**Иначе:** варпим предыдущий результат локально (0 API calls)

In [ ]:
def should_regenerate(
    prev_mask: Optional[np.ndarray],
    curr_mask: Optional[np.ndarray],
    flow: Optional[np.ndarray],
    flow_threshold: float,
    iou_threshold: float
) -> Tuple[bool, str]:
    """
    Определяет нужна ли regeneration.
    Возвращает (нужна?, причина).
    """
    # Нет предыдущего результата → нужна генерация
    if prev_mask is None:
        return True, 'no_previous_mask'

    # Нет текущей маски → объект пропал
    if curr_mask is None:
        return False, 'object_not_visible'

    # Проверяем flow magnitude
    if flow is not None:
        # Flow только в области маски — нас интересует движение объекта
        mask_region = curr_mask > 0
        if mask_region.sum() > 0:
            flow_in_mask = flow[mask_region]
            mag = float(np.sqrt(flow_in_mask[:, 0]**2 + flow_in_mask[:, 1]**2).mean())
            if mag > flow_threshold:
                return True, f'flow_magnitude={mag:.1f}'

    # Проверяем IoU масок
    iou = compute_mask_iou(prev_mask, curr_mask)
    if iou < iou_threshold:
        return True, f'low_iou={iou:.2f}'

    return False, 'warp'


print('✅ Логика regeneration готова')

## 12. Diffusion Inpainting через Replicate API

Ключевой трюк: **не передаём весь кадр** — только bbox crop с небольшим padding.
Это означает что модель видит только объект + чуть-чуть окружения.
Фон вообще не трогается.

После получения результата — вставляем crop обратно в оригинальный кадр.

In [ ]:
def frame_to_base64(frame: np.ndarray) -> str:
    """Конвертирует BGR numpy frame в base64 PNG строку."""
    pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    buffer = io.BytesIO()
    pil.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def mask_to_base64(mask: np.ndarray) -> str:
    """Конвертирует бинарную маску в base64 PNG."""
    pil = Image.fromarray(mask)
    buffer = io.BytesIO()
    pil.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')


def crop_frame_and_mask(
    frame: np.ndarray,
    mask: np.ndarray,
    bbox: Tuple[int, int, int, int],
    padding: float
) -> Tuple[np.ndarray, np.ndarray, Tuple]:
    """
    Вырезает bbox + padding из кадра и маски.
    Возвращает (crop_frame, crop_mask, padded_bbox).
    """
    padded_bbox = add_bbox_padding(bbox, frame.shape, padding)
    x1, y1, x2, y2 = padded_bbox
    return frame[y1:y2, x1:x2], mask[y1:y2, x1:x2], padded_bbox


def call_inpainting_api_image(
    frame_crop: np.ndarray,
    mask_crop: np.ndarray,
    prompt: str,
    negative_prompt: str,
    seed: int
) -> np.ndarray:
    """
    Вызывает SDXL Inpainting на одном кадре через Replicate.
    Возвращает BGR numpy изображение того же размера что crop.
    """
    original_size = (frame_crop.shape[1], frame_crop.shape[0])  # (w, h)

    output = replicate.run(
        'stability-ai/stable-diffusion-inpainting:95b7223104132402a9ae91cc677285bc5eb997834bd2349fa486f53910fd68b3',
        input={
            'image': f'data:image/png;base64,{frame_to_base64(frame_crop)}',
            'mask': f'data:image/png;base64,{mask_to_base64(mask_crop)}',
            'prompt': prompt,
            'negative_prompt': negative_prompt,
            'seed': seed,
            'num_inference_steps': 30,
            'guidance_scale': 7.5,
        }
    )

    # Получаем результат и конвертируем обратно в BGR numpy
    import urllib.request
    result_url = output[0] if isinstance(output, list) else output
    with urllib.request.urlopen(result_url) as response:
        result_bytes = response.read()

    result_pil = Image.open(io.BytesIO(result_bytes)).convert('RGB')
    result_pil = result_pil.resize(original_size, Image.LANCZOS)
    result_bgr = cv2.cvtColor(np.array(result_pil), cv2.COLOR_RGB2BGR)

    return result_bgr


def call_inpainting_api_video(
    frame_crops: List[np.ndarray],
    mask_crops: List[np.ndarray],
    prompt: str,
    negative_prompt: str,
    seed: int
) -> List[np.ndarray]:
    """
    Вызывает Wan2.1-Fun-Inpaint на чанке кадров через Replicate.
    Возвращает список BGR кадров.
    """
    import tempfile, urllib.request

    original_size = (frame_crops[0].shape[1], frame_crops[0].shape[0])

    # Собираем мини-видео из crops
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp_video_path = f.name
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp_mask_path = f.name

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_v = cv2.VideoWriter(tmp_video_path, fourcc, 16, original_size)
    out_m = cv2.VideoWriter(tmp_mask_path, fourcc, 16, original_size)

    for crop, mask in zip(frame_crops, mask_crops):
        out_v.write(crop)
        out_m.write(cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR))

    out_v.release()
    out_m.release()

    # Загружаем в Replicate
    with open(tmp_video_path, 'rb') as f:
        video_data = f.read()
    with open(tmp_mask_path, 'rb') as f:
        mask_data = f.read()

    output = replicate.run(
        'wan-video/wan2.1-fun-inpaint',
        input={
            'video': f'data:video/mp4;base64,{base64.b64encode(video_data).decode()}',
            'mask': f'data:video/mp4;base64,{base64.b64encode(mask_data).decode()}',
            'prompt': prompt,
            'negative_prompt': negative_prompt,
            'seed': seed,
            'num_inference_steps': 30,
        }
    )

    # Декодируем результирующее видео
    result_url = output if isinstance(output, str) else output[0]
    with urllib.request.urlopen(result_url) as response:
        result_bytes = response.read()

    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        f.write(result_bytes)
        result_path = f.name

    result_frames = []
    cap = cv2.VideoCapture(result_path)
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, original_size)
        result_frames.append(frame)
    cap.release()

    os.unlink(tmp_video_path)
    os.unlink(tmp_mask_path)
    os.unlink(result_path)

    return result_frames


def call_inpainting(
    frames_or_frame,
    masks_or_mask,
    bbox: Tuple,
    cfg: Config
):
    """
    Единая точка входа для inpainting.
    Выбирает картиночную или видео модель в зависимости от cfg.use_video_model.
    """
    if cfg.use_video_model and isinstance(frames_or_frame, list):
        # Режим видео: несколько кадров
        crops, mask_crops, padded_bboxes = [], [], []
        for frame, mask in zip(frames_or_frame, masks_or_mask):
            fc, mc, pb = crop_frame_and_mask(frame, mask, bbox, cfg.BBOX_PADDING)
            crops.append(fc)
            mask_crops.append(mc)
            padded_bboxes.append(pb)

        result_crops = call_inpainting_api_video(
            crops, mask_crops, cfg.replacement_prompt, cfg.negative_prompt, cfg.seed
        )
        return result_crops, padded_bboxes
    else:
        # Режим картинки: один кадр
        frame = frames_or_frame if not isinstance(frames_or_frame, list) else frames_or_frame[0]
        mask = masks_or_mask if not isinstance(masks_or_mask, list) else masks_or_mask[0]
        fc, mc, pb = crop_frame_and_mask(frame, mask, bbox, cfg.BBOX_PADDING)

        result_crop = call_inpainting_api_image(
            fc, mc, cfg.replacement_prompt, cfg.negative_prompt, cfg.seed
        )
        return [result_crop], [pb]


print('✅ Inpainting API функции готовы')

## 13. Compositing: вставка результата обратно в кадр

Берём сгенерированный crop и вставляем его точно по bbox координатам.
Используем маску для плавного blending по краям объекта.

In [ ]:
def composite_into_frame(
    original_frame: np.ndarray,
    generated_crop: np.ndarray,
    mask: np.ndarray,
    padded_bbox: Tuple[int, int, int, int],
    feather_radius: int = 5
) -> np.ndarray:
    """
    Вставляет generated_crop обратно в original_frame.

    Алгоритм:
    1. Crop маски по padded_bbox
    2. Размываем края маски (feathering) для плавного перехода
    3. Alpha blending: result = mask * generated + (1 - mask) * original
    """
    result = original_frame.copy()
    x1, y1, x2, y2 = padded_bbox
    h_crop, w_crop = y2 - y1, x2 - x1

    # Crop маски по bbox
    mask_crop = mask[y1:y2, x1:x2]

    # Resize generated_crop если размеры не совпадают (после API могут отличаться)
    if generated_crop.shape[:2] != (h_crop, w_crop):
        generated_crop = cv2.resize(generated_crop, (w_crop, h_crop))

    # Feathering: размываем маску по краям
    if feather_radius > 0:
        kernel_size = feather_radius * 2 + 1
        alpha = cv2.GaussianBlur(
            mask_crop.astype(np.float32) / 255.0,
            (kernel_size, kernel_size), 0
        )
    else:
        alpha = mask_crop.astype(np.float32) / 255.0

    alpha_3ch = alpha[:, :, np.newaxis]  # (H, W, 1) для broadcast

    # Alpha blending
    original_crop = original_frame[y1:y2, x1:x2].astype(np.float32)
    gen_crop_float = generated_crop.astype(np.float32)

    blended = alpha_3ch * gen_crop_float + (1.0 - alpha_3ch) * original_crop
    result[y1:y2, x1:x2] = blended.astype(np.uint8)

    return result


print('✅ Compositing функция готова')

## 14. Главный пайплайн

Собираем всё вместе. Для каждого кадра:

```
1. Shot cut? → переинициализация трекера + re-detect
2. Объект виден?
   → нет: пропускаем, копируем оригинальный кадр
   → да: продолжаем
3. Объект только что появился после пропуска?
   → re-ID через CLIP
   → подтверждён → принудительная regeneration
4. should_regenerate?
   → да: API вызов (по bbox crop)
   → нет: warp предыдущего результата
5. composite результата в кадр
```

In [ ]:
def run_pipeline(state: VideoState, cfg: Config) -> VideoState:
    """
    Главный пайплайн. Обрабатывает все кадры и заполняет state.output_frames.
    """
    frames = state.frames
    n_frames = len(frames)
    shot_boundaries_set = set(state.shot_boundaries)

    # Состояние трекинга
    prev_mask = None
    prev_generated_crop = None
    prev_padded_bbox = None
    prev_bbox = None
    was_visible = False
    frames_missing_count = 0
    api_calls_count = 0

    # Трекинг масок по сценам
    all_masks = [None] * n_frames
    all_bboxes = [None] * n_frames

    # ── Шаг 1: Трекинг масок по всему видео (SAM2) ────────────────────────────
    print('\n🔍 Шаг 1: Трекинг масок через SAM2...')

    # Обрабатываем каждую сцену отдельно
    boundaries = sorted(state.shot_boundaries)
    for scene_idx, scene_start in enumerate(tqdm(boundaries, desc='Сцены')):
        scene_end = boundaries[scene_idx + 1] if scene_idx + 1 < len(boundaries) else n_frames

        # Детектируем объект на первом кадре сцены
        first_frame = frames[scene_start]
        print(f'\n  Сцена {scene_idx+1}: кадры [{scene_start}:{scene_end}]')

        bbox = detect_object(gdino_model, first_frame, cfg.detection_prompt, cfg.device)

        if bbox is None:
            print(f'  ⚠️ Объект не найден в сцене {scene_idx+1}, пропускаем')
            continue

        # Трекаем маски через SAM2
        scene_masks, scene_bboxes = track_masks_in_scene(
            sam2_predictor, frames, bbox, scene_start, scene_end
        )

        for i, (mask, b) in enumerate(zip(scene_masks, scene_bboxes)):
            all_masks[scene_start + i] = mask
            all_bboxes[scene_start + i] = b

    state.masks = all_masks
    state.bboxes = all_bboxes

    # ── Шаг 2: Optical flow + Inpainting + Compositing ───────────────────────
    print('\n🎨 Шаг 2: Inpainting и compositing...')

    for i in tqdm(range(n_frames), desc='Обработка кадров'):
        frame = frames[i]
        curr_mask = all_masks[i]
        curr_bbox = all_bboxes[i]
        is_shot_cut = i in shot_boundaries_set and i > 0
        object_visible = curr_mask is not None and curr_bbox is not None

        state.object_visible.append(object_visible)

        # Объект не виден → копируем оригинальный кадр
        if not object_visible:
            state.output_frames.append(frame.copy())
            frames_missing_count += 1

            if frames_missing_count > cfg.MAX_FRAMES_MISSING:
                # Сбрасываем трекинг — объект пропал надолго
                prev_mask = None
                prev_generated_crop = None
                state.last_known_appearance = None
                print(f'  ⚠️ Кадр {i}: объект пропал на {frames_missing_count} кадров, сброс трекинга')

            was_visible = False
            continue

        # Объект только что появился после пропуска
        reappeared = not was_visible and frames_missing_count > 0
        frames_missing_count = 0
        was_visible = True

        # Re-identification если объект вернулся
        force_regen = False
        if reappeared and state.last_known_appearance is not None:
            confirmed = is_same_object(
                clip_model, clip_processor,
                frame, curr_bbox,
                state.last_known_appearance,
                cfg.REID_CLIP_THRESHOLD,
                cfg.device
            )
            if confirmed:
                print(f'  ✅ Кадр {i}: объект переопознан, принудительная regeneration')
                force_regen = True
            else:
                print(f'  ⚠️ Кадр {i}: объект не подтверждён (другой объект?)')

        # Обновляем CLIP appearance
        state.last_known_appearance = get_object_embedding(
            clip_model, clip_processor, frame, curr_bbox, cfg.device
        )

        # Вычисляем optical flow (если есть предыдущий кадр)
        flow = None
        if i > 0 and not is_shot_cut:
            flow = compute_optical_flow(raft_model, frames[i-1], frame, cfg.device)
            state.flows.append(flow)
        else:
            state.flows.append(None)

        # Решаем: regenerate или warp
        need_regen, reason = should_regenerate(
            prev_mask, curr_mask, flow,
            cfg.FLOW_THRESHOLD, cfg.IOU_THRESHOLD
        )
        need_regen = need_regen or is_shot_cut or force_regen or (i == 0)

        if need_regen:
            # ── REGENERATION: вызов API ───────────────────────────────────────
            reason_str = 'shot_cut' if is_shot_cut else ('reappeared' if force_regen else reason)
            print(f'  🔄 Кадр {i}: regeneration [{reason_str}]')

            result_crops, padded_bboxes = call_inpainting(
                frame, curr_mask, curr_bbox, cfg
            )

            generated_crop = result_crops[0]
            padded_bbox = padded_bboxes[0]
            state.generated_objects[i] = (generated_crop, padded_bbox)
            api_calls_count += 1

        else:
            # ── WARP: используем предыдущий результат ────────────────────────
            if prev_generated_crop is not None and flow is not None:
                x1, y1, x2, y2 = prev_padded_bbox
                flow_crop = flow[y1:y2, x1:x2]
                generated_crop = warp_image_with_flow(prev_generated_crop, flow_crop)
                padded_bbox = prev_padded_bbox
            else:
                # Fallback: используем предыдущий crop без варпинга
                generated_crop = prev_generated_crop
                padded_bbox = prev_padded_bbox

        # ── Compositing ───────────────────────────────────────────────────────
        if generated_crop is not None and padded_bbox is not None:
            output_frame = composite_into_frame(
                frame, generated_crop, curr_mask, padded_bbox
            )
        else:
            output_frame = frame.copy()

        state.output_frames.append(output_frame)

        # Обновляем состояние для следующего кадра
        prev_mask = curr_mask
        prev_generated_crop = generated_crop
        prev_padded_bbox = padded_bbox
        prev_bbox = curr_bbox

    print(f'\n✅ Готово! API вызовов: {api_calls_count} из {n_frames} кадров')
    return state


print('✅ Пайплайн определён')

## 15. Запуск пайплайна

In [ ]:
# Запускаем!
state = run_pipeline(state, cfg)

print(f'Обработано кадров: {len(state.output_frames)}')
print(f'Кадров с объектом: {sum(state.object_visible)}')
print(f'Shot cuts: {state.shot_boundaries}')

## 16. Сборка итогового видео

In [ ]:
def assemble_video(
    frames: List[np.ndarray],
    output_path: str,
    fps: float,
    frame_size: Tuple[int, int]
):
    """
    Собирает список кадров в MP4 видео.
    frame_size: (width, height)
    """
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

    for frame in tqdm(frames, desc='Записываем видео'):
        # Убеждаемся что размер правильный
        if (frame.shape[1], frame.shape[0]) != frame_size:
            frame = cv2.resize(frame, frame_size)
        writer.write(frame)

    writer.release()
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'✅ Видео сохранено: {output_path} ({size_mb:.1f} MB)')


assemble_video(
    state.output_frames,
    cfg.output_video_path,
    fps=cfg.target_fps or state.fps,
    frame_size=state.frame_size
)

# Сохраняем отдельные кадры для отладки
save_frames(state.output_frames, cfg.output_frames_dir)
print(f'✅ Кадры сохранены в {cfg.output_frames_dir}')

## 17. Визуализация

Инструменты для отладки пайплайна.

In [ ]:
def visualize_mask_on_frame(
    frame: np.ndarray,
    mask: Optional[np.ndarray],
    bbox: Optional[Tuple] = None,
    color: Tuple = (0, 255, 0),
    alpha: float = 0.4
) -> np.ndarray:
    """Накладывает маску и bbox на кадр."""
    vis = frame.copy()

    if mask is not None:
        overlay = vis.copy()
        overlay[mask > 0] = color
        vis = cv2.addWeighted(vis, 1 - alpha, overlay, alpha, 0)

    if bbox is not None:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

    return vis


def visualize_flow(flow: np.ndarray) -> np.ndarray:
    """Конвертирует optical flow в HSV цветовую карту."""
    h, w = flow.shape[:2]
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((h, w, 3), dtype=np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2  # Hue = направление
    hsv[..., 1] = 255                     # Saturation = max
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)  # Value = magnitude
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


def show_comparison(
    state: VideoState,
    frame_indices: List[int],
    show_flow: bool = True
):
    """Показывает сравнение оригинал / маска / результат для выбранных кадров."""
    n_cols = 4 if show_flow else 3
    fig, axes = plt.subplots(len(frame_indices), n_cols, figsize=(5 * n_cols, 4 * len(frame_indices)))

    if len(frame_indices) == 1:
        axes = [axes]

    for row, idx in enumerate(frame_indices):
        orig = cv2.cvtColor(state.frames[idx], cv2.COLOR_BGR2RGB)
        mask = state.masks[idx] if idx < len(state.masks) else None
        bbox = state.bboxes[idx] if idx < len(state.bboxes) else None
        out = cv2.cvtColor(state.output_frames[idx], cv2.COLOR_BGR2RGB) if idx < len(state.output_frames) else orig

        mask_vis = cv2.cvtColor(
            visualize_mask_on_frame(state.frames[idx], mask, bbox),
            cv2.COLOR_BGR2RGB
        )

        axes[row][0].imshow(orig)
        axes[row][0].set_title(f'Кадр {idx}: Оригинал')
        axes[row][0].axis('off')

        axes[row][1].imshow(mask_vis)
        axes[row][1].set_title('Маска SAM2')
        axes[row][1].axis('off')

        axes[row][2].imshow(out)
        axes[row][2].set_title('Результат')
        axes[row][2].axis('off')

        if show_flow and idx < len(state.flows) and state.flows[idx] is not None:
            flow_vis = cv2.cvtColor(visualize_flow(state.flows[idx]), cv2.COLOR_BGR2RGB)
            axes[row][3].imshow(flow_vis)
            axes[row][3].set_title('Optical Flow')
            axes[row][3].axis('off')
        elif show_flow:
            axes[row][3].axis('off')

    plt.tight_layout()
    plt.show()


def show_shot_boundaries(state: VideoState, n_frames_context: int = 1):
    """Показывает кадры вокруг каждого shot cut."""
    for cut_idx in state.shot_boundaries[1:]:
        indices = list(range(
            max(0, cut_idx - n_frames_context),
            min(len(state.frames), cut_idx + n_frames_context + 1)
        ))
        print(f'\n--- Shot cut на кадре {cut_idx} ---')
        show_comparison(state, indices, show_flow=False)


# Показываем несколько кадров
sample_indices = list(range(0, min(len(state.output_frames), 5)))
show_comparison(state, sample_indices)

# Показываем shot cuts если есть
if len(state.shot_boundaries) > 1:
    show_shot_boundaries(state)

## 18. Статистика пайплайна

In [ ]:
def print_pipeline_stats(state: VideoState, cfg: Config):
    n = len(state.frames)
    n_visible = sum(state.object_visible)
    n_regen = len(state.generated_objects)
    n_warp = n_visible - n_regen
    n_cuts = len(state.shot_boundaries) - 1

    print('=' * 50)
    print('📊 СТАТИСТИКА ПАЙПЛАЙНА')
    print('=' * 50)
    print(f'Всего кадров:          {n}')
    print(f'Кадров с объектом:     {n_visible} ({100*n_visible/n:.1f}%)')
    print(f'Shot cuts:             {n_cuts}')
    print(f'API вызовов (regen):   {n_regen}')
    print(f'Warp кадров:           {n_warp}')
    if n_visible > 0:
        print(f'API вызовов на кадр:   {n_regen/n_visible:.1%}')
    print(f'Порог flow:            {cfg.FLOW_THRESHOLD} px')
    print(f'Порог IoU:             {cfg.IOU_THRESHOLD}')
    print(f'Порог re-ID:           {cfg.REID_CLIP_THRESHOLD}')
    print('=' * 50)


print_pipeline_stats(state, cfg)